# 1장. 랭체인 기초

In [ ]:
# ⏱ 패키지 설치 (1~2분 소요)
# langchain: LLM 애플리케이션 프레임워크
# langchain-ollama: Ollama 모델 연동 (ChatOllama, OllamaLLM)
# langchain-community: 서드파티 통합 도구
!pip install -q langchain langchain-ollama langchain-community

In [ ]:
# OpenAI API는 사용하지 않습니다 (Ollama 사용)
# from google.colab import userdata
# import os

# os.environ['OPENAI_API_KEY']=userdata.get('OPENAI_API_KEY')

In [ ]:
# ⏱ Ollama 설치 및 모델 다운로드 (3~5분 소요)
import subprocess
import time

!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

subprocess.Popen(['ollama', 'serve'])
time.sleep(3)

!ollama pull llama3.2

## 코드 1-1 기본 LLM 호출

In [ ]:
from langchain_ollama import OllamaLLM

model = OllamaLLM(model="llama3.2")

model.invoke("The sky is")

## 코드 1-2 채팅 모델 호출

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

model = ChatOllama(model="llama3.2")
prompt = [HumanMessage("What is the capital of France?")]

model.invoke(prompt)


## 코드 1-3 시스템 메시지를 적용한 채팅 모델 호출

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_ollama import ChatOllama

model = ChatOllama(model="llama3.2")
system_msg = SystemMessage(
    '''You are a helpful assistant that responds to questions with three
        exclamation marks.'''
)
human_msg = HumanMessage('What is the capital of France?')

model.invoke([system_msg, human_msg])

## 코드 1-4 프롬프트 템플릿 적용

In [ ]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate.from_template("""Answer the question based on the
    context below. If the question cannot be answered using the information
    provided, answer with "I don't know".

Context: {context}

Question: {question}

Answer: """)

template.invoke({
    "context": """The most recent advancements in NLP are being driven by Large
        Language Models (LLMs). These models outperform their smaller
        counterparts and have become invaluable for developers who are creating
        applications with NLP capabilities. Developers can tap into these
        models through Hugging Face's `transformers` library, or by utilizing
        OpenAI and Cohere's offerings through the `openai` and `cohere`
        libraries, respectively.""",
    "question": "Which model providers offer LLMs?"
})

## 코드 1-5 동적 프롬프트

In [ ]:
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate

template = PromptTemplate.from_template("""Answer the question based on the
    context below. If the question cannot be answered using the information
    provided, answer with "I don't know".

Context: {context}

Question: {question}

Answer: """)

model = OllamaLLM(model="llama3.2")

prompt = template.invoke({
    "context": """The most recent advancements in NLP are being driven by Large
        Language Models (LLMs). These models outperform their smaller
        counterparts and have become invaluable for developers who are creating
        applications with NLP capabilities. Developers can tap into these
        models through Hugging Face's `transformers` library, or by utilizing
        OpenAI and Cohere's offerings through the `openai` and `cohere`
        libraries, respectively.""",
    "question": "Which model providers offer LLMs?"
})

completion = model.invoke(prompt)

print(completion)

## 코드 1-6 역할에 따른 동적 프롬프트

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
template = ChatPromptTemplate.from_messages([
    ('system', '''Answer the question based on the context below. If the
        question cannot be answered using the information provided, answer with
        "I don\'t know".'''),
    ('human', 'Context: {context}'),
    ('human', 'Question: {question}'),
])

template.invoke({
    "context": """The most recent advancements in NLP are being driven by Large
        Language Models (LLMs). These models outperform their smaller
        counterparts and have become invaluable for developers who are creating
        applications with NLP capabilities. Developers can tap into these
        models through Hugging Face's `transformers` library, or by utilizing
        OpenAI and Cohere's offerings through the `openai` and `cohere`
        libraries, respectively.""",
    "question": "Which model providers offer LLMs?"
})

## 코드 1-7 두 개의 동적 프롬프트를 적용한 호출

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate.from_messages([
    ('system', '''Answer the question based on the context below. If the
        question cannot be answered using the information provided, answer
        with "I don\'t know".'''),
    ('human', 'Context: {context}'),
    ('human', 'Question: {question}'),
])

model = ChatOllama(model="llama3.2")

prompt = template.invoke({
    "context": """The most recent advancements in NLP are being driven by
        Large Language Models (LLMs). These models outperform their smaller
        counterparts and have become invaluable for developers who are creating
        applications with NLP capabilities. Developers can tap into these
        models through Hugging Face's `transformers` library, or by utilizing
        OpenAI and Cohere's offerings through the `openai` and `cohere`
        libraries, respectively.""",
    "question": "Which model providers offer LLMs?"
})

model.invoke(prompt)

## 코드 1-8 JSON 형식 출력 요청

In [ ]:
from langchain_ollama import ChatOllama
from pydantic import BaseModel

class AnswerWithJustification(BaseModel):
    '''An answer to the user's question along with justification for the
        answer.'''
    answer: str
    justification: str

llm = ChatOllama(model="llama3.2", temperature=0)
structured_llm = llm.with_structured_output(AnswerWithJustification)

result = structured_llm.invoke("What weighs more, a pound of bricks or a pound of feathers?")

print(result.model_dump_json())


## 코드 1-9 랭체인의 CSV 출력 파서

In [ ]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
parser = CommaSeparatedListOutputParser()
items = parser.invoke("apple, banana, cherry")
print(items)

## 코드 1-10 랭체인의 공통 인터페이스 예시

In [ ]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="llama3.2")

completion = model.invoke('Hi there!')
print(completion)

completions = model.batch(['Hi there!', 'Bye!'])
print(completions)

for token in model.stream('Bye!'):
    print(token, end="| ", flush=True)

## 코드 1-11 명령형 구성 예시

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import chain

template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        ("human", "{question}"),
    ]
)

model = ChatOllama(model="llama3.2")

@chain
def chatbot(values):
    prompt = template.invoke(values)
    return model.invoke(prompt)

response = chatbot.invoke({"question": "Which model providers offer LLMs?"})
print(response.content)

## 코드 1-12 명령형 구성을 사용한 스트리밍 호출 예시

In [ ]:
@chain
def chatbot(values):
    prompt = template.invoke(values)
    for token in model.stream(prompt):
        yield token

for part in chatbot.stream({
    "question": "Which model providers offer LLMs?"
}):
    print(part.content, end="", flush=True)


## 코드 1-13 명령형 구성을 사용한 비동기 실행

In [ ]:
@chain
async def chatbot(values):
    prompt = await template.ainvoke(values)
    return await model.ainvoke(prompt)

await chatbot.ainvoke({"question": "Which model providers offer LLMs?"})


## 코드 1-14 선언형 구성 예시

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        ("human", "{question}"),
    ]
)

model = ChatOllama(model="llama3.2")

chatbot = template | model

response = chatbot.invoke({"question": "Which model providers offer LLMs?"})
print(response.content)

for part in chatbot.stream({"question": "Which model providers offer LLMs?"}):
    print(part.content, end="", flush=True)

## 코드 1-15 선언형 구성을 사용한 스트리밍 호출 예시

In [ ]:
chatbot = template | model

for part in chatbot.stream({
    "question": "Which model providers offer LLMs?"
}):
    print(part)


## 코드 1-16 선언형 구성을 사용한 비동기 실행

In [ ]:
chatbot = template | model

await chatbot.ainvoke({
    "question": "Which model providers offer LLMs?"
})
